## Modeling the agent
1. Feature selection: remove the columns I will not use (for example, the indexing, lat, longitude, and name of the station)
2. Train the models with resampling
3. .....


### Libraries and tools to ues




### Preparation on google colab

!pip install -U ipython

In [ ]:
"""
%load_ext autoreload
%autoreload 2

#Clone the repository
import os
if not os.path.exists("your-project"):
    !git clone https://github.com/ras112git/predictive_modeling_and_mobility_forecasting.git
else:
    print("Repo already cloned.")


%cd predictive_modeling_and_mobility_forecasting

# Install dependencies
!pip install -r requirements.txt

import sys
sys.path.append(os.getcwd())
"""


### Preparation of the venvironment (as on the other notebooks)

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")


Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Automatically clean the data

In [2]:
from src.model import *

import pandas as pd

# Start by importing the data
train_path = Path('data/raw/dataset_train.csv')
df_train = pd.read_csv(train_path)

In [3]:
# I will clean it heare, as if I cleaned it and saved it as csv, then the categories will be missed

from src.data_cleaning import clean_data

df_train = clean_data(dataset= df_train, 
                      is_train = True, 
                      categorize_station = True) # If I put false here, it splits the station data in bolean

df_train

,id,datetime,station_number,name,lat,lng,bikes,hour,minute,dayofweek,...,hour_cos,is_holiday,avg_bikes_per_station,temperature_2m,apparent_temperature,precipitation,snowfall,wind_speed_10m,cloud_cover,relative_humidity_2m
0,2024-09-03 17:30:00_32000,2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,25,17,30,1,...,-0.258819,0,9.258033,28.8,27.7,0.0,0.0,8.4,6,32
1,2024-09-03 17:30:00_32001,2024-09-03 17:30:00,32001,Hoher Markt,48.210666,16.372983,14,17,30,1,...,-0.258819,0,16.898142,28.8,27.7,0.0,0.0,8.4,6,32
2,2024-09-03 17:30:00_32002,2024-09-03 17:30:00,32002,Oper,48.202683,16.369702,9,17,30,1,...,-0.258819,0,14.598361,28.8,27.7,0.0,0.0,8.4,6,32
3,2024-09-03 17:30:00_32003,2024-09-03 17:30:00,32003,Volksgarten,48.206516,16.360400,3,17,30,1,...,-0.258819,0,7.415082,28.8,27.7,0.0,0.0,8.4,6,32
4,2024-09-03 17:30:00_32004,2024-09-03 17:30:00,32004,Taborstraße U2,48.219522,16.382218,5,17,30,1,...,-0.258819,0,12.433880,28.8,27.7,0.0,0.0,8.4,6,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2150245,2025-03-13 08:00:00_32275,2025-03-13 08:00:00,32275,eLastenräder - Am langen Felde,48.250224,16.450650,0,8,0,3,...,-0.500000,0,0.675519,10.1,7.2,0.5,0.0,16.4,100,88
2150246,2025-03-13 08:00:00_32277,2025-03-13 08:00:00,32277,eLastenräder - Bruno-Marek-Allee 6,48.227914,16.391516,6,8,0,3,...,-0.500000,0,3.513661,10.1,7.2,0.5,0.0,16.4,100,88
2150247,2025-03-13 08:00:00_32278,2025-03-13 08:00:00,32278,eLastenräder - Am Tabor 23,48.224598,16.392090,3,8,0,3,...,-0.500000,0,2.462186,10.1,7.2,0.5,0.0,16.4,100,88
2150248,2025-03-13 08:00:00_32280,2025-03-13 08:00:00,32280,ALF Mobility-Point,48.251355,16.452810,5,8,0,3,...,-0.500000,0,4.515519,10.1,7.2,0.5,0.0,16.4,100,88


### Feature selection

In [4]:
# Split the features

TARGET = "bikes"

x_train, y_train = data_split(df_train,target=TARGET)
#print(X_train)


In [5]:
#Make sure that the data is ordered by datetime
assert x_train['datetime'].is_monotonic_increasing

DROP_COLS = ["id", "datetime", "name", "lat", "lng"] #

x_train = prepare_features(X = x_train, drop_cols=DROP_COLS)

In [6]:
# featurce_performance = feature_selection_report(x_train,y_train)

# print("features:", featurce_performance)
# print("train data shape:", x_train.shape)


In [7]:
x_train

,station_number,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos,is_holiday,avg_bikes_per_station,temperature_2m,apparent_temperature,precipitation,snowfall,wind_speed_10m,cloud_cover,relative_humidity_2m
0,32000,17,30,1,9,0,-0.965926,-0.258819,0,9.258033,28.8,27.7,0.0,0.0,8.4,6,32
1,32001,17,30,1,9,0,-0.965926,-0.258819,0,16.898142,28.8,27.7,0.0,0.0,8.4,6,32
2,32002,17,30,1,9,0,-0.965926,-0.258819,0,14.598361,28.8,27.7,0.0,0.0,8.4,6,32
3,32003,17,30,1,9,0,-0.965926,-0.258819,0,7.415082,28.8,27.7,0.0,0.0,8.4,6,32
4,32004,17,30,1,9,0,-0.965926,-0.258819,0,12.433880,28.8,27.7,0.0,0.0,8.4,6,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2150245,32275,8,0,3,3,0,0.866025,-0.500000,0,0.675519,10.1,7.2,0.5,0.0,16.4,100,88
2150246,32277,8,0,3,3,0,0.866025,-0.500000,0,3.513661,10.1,7.2,0.5,0.0,16.4,100,88
2150247,32278,8,0,3,3,0,0.866025,-0.500000,0,2.462186,10.1,7.2,0.5,0.0,16.4,100,88
2150248,32280,8,0,3,3,0,0.866025,-0.500000,0,4.515519,10.1,7.2,0.5,0.0,16.4,100,88


### Benchmarking different methods

In [8]:
#I first need to create all the models that I will asses
model_grid = get_model_grid(Feature_df=x_train)

In [9]:
#Then i train them with the default folds
benchmark_results = benchmark_models(X=x_train,y=y_train,models=model_grid,sample_size=200_000)

c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 2 is smaller than n_iter=8. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 2 is smaller than n_iter=8. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 2 is smaller than n_iter=8. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


  Featureless fold 1/3: RMSE=9.886  MAE=7.731 RMSLE=1.010
  Featureless fold 2/3: RMSE=8.842  MAE=6.963 RMSLE=0.890
  Featureless fold 3/3: RMSE=9.113  MAE=7.227 RMSLE=0.915
  decision_tree fold 1/3: RMSE=7.373  MAE=5.507 RMSLE=0.738
  decision_tree fold 2/3: RMSE=7.076  MAE=5.446 RMSLE=0.777
  decision_tree fold 3/3: RMSE=6.582  MAE=4.909 RMSLE=0.695
  random_forest fold 1/3: RMSE=6.991  MAE=5.251 RMSLE=0.696
  random_forest fold 2/3: RMSE=6.156  MAE=4.791 RMSLE=0.680


KeyboardInterrupt: 

In [ ]:
print(benchmark_results)

In [ ]:
#Finally we print to see the higher performing
from src.model import plot_benchmark

fig, ax = plot_benchmark(benchmark_results)

### Seeing those results, we now select one of the models and train it
This time with more hyperparameters, during a longer period of time, and with a more overall complex resampling strategy.

In [10]:
# Remember to pass a model that handles categorical data, if there will be categorical data in pd version
model_name = "lightgbm"


#model_name = benchmark_results.iloc[0]['model']

best_params, search = train_hyperparameters(
    X=x_train, y=y_train, model_name=model_name, n_iter=40, log_target=True, sample_size=100_000
)

#sample_size to be specified, n_iter

[train_hyperparameters] model=lightgbm  log_target=True  rows=100,000  features=17  n_iter=40  cv_splits=3  total_fits=120  scoring=rmsle
[train_hyperparameters] param grid keys: ['regressor__colsample_bytree', 'regressor__learning_rate', 'regressor__max_depth', 'regressor__min_child_samples', 'regressor__n_estimators', 'regressor__num_leaves', 'regressor__reg_alpha', 'regressor__reg_lambda', 'regressor__subsample']

[  1/40] mean_rmsle=0.6597  std=0.0618  folds=['0.617', '0.747', '0.615']  time=35.0s  params={'regressor__subsample': 0.9, 'regressor__reg_lambda': 10.0, 'regressor__reg_alpha': 0.01, 'regressor__num_leaves': 511, 'regressor__n_estimators': 1000, 'regressor__min_child_samples': 5, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.05, 'regressor__colsample_bytree': 0.6}
[  2/40] mean_rmsle=0.6627  std=0.0664  folds=['0.619', '0.757', '0.613']  time=23.3s  params={'regressor__subsample': 0.6, 'regressor__reg_lambda': 10.0, 'regressor__reg_alpha': 0.01, 'regressor__nu

In [11]:
print(best_params)

{'regressor__subsample': 0.8, 'regressor__reg_lambda': 0.1, 'regressor__reg_alpha': 0, 'regressor__num_leaves': 127, 'regressor__n_estimators': 200, 'regressor__min_child_samples': 100, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.01, 'regressor__colsample_bytree': 0.7}


In [12]:
# best_params = {'regressor__subsample': 0.8, 'regressor__reg_lambda': 0.1, 'regressor__reg_alpha': 0, 'regressor__num_leaves': 127, 'regressor__n_estimators': 200, 'regressor__min_child_samples': 100, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.01, 'regressor__colsample_bytree': 0.7}

model = train_final_model(
    X=x_train, y=y_train, model_name=model_name,
    best_params=best_params, log_target=True,   # optionally sample_size=...
)

[train_final_model] refitting lightgbm on rows=2,150,250, features=17, log_target=True...
[train_final_model] params: {'regressor__subsample': 0.8, 'regressor__reg_lambda': 0.1, 'regressor__reg_alpha': 0, 'regressor__num_leaves': 127, 'regressor__n_estimators': 200, 'regressor__min_child_samples': 100, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.01, 'regressor__colsample_bytree': 0.7}


In [13]:
print(model)
print(search)

TransformedTargetRegressor(check_inverse=False, func=<ufunc 'log1p'>,
                           inverse_func=<ufunc 'expm1'>,
                           regressor=LGBMRegressor(colsample_bytree=0.7,
                                                   learning_rate=0.01,
                                                   max_depth=8,
                                                   min_child_samples=100,
                                                   n_estimators=200, n_jobs=-2,
                                                   num_leaves=127,
                                                   objective='regression',
                                                   random_state=42, reg_alpha=0,
                                                   reg_lambda=0.1,
                                                   subsample=0.8, verbose=-1))
namespace(best_params_={'regressor__subsample': 0.8, 'regressor__reg_lambda': 0.1, 'regressor__reg_alpha': 0, 'regressor__num_leaves': 127, 'r

In [14]:
evaluate(X = x_train,y = y_train,model=model)

[ 7.16276425 12.65297505  8.88162355 ...  2.3719451   3.6434086
  1.00042298]
MAE:   4.771
RMSE:  6.504
RMSLE: 0.595


In [15]:
import joblib
# Save the model
joblib.dump(model, f'models/{model_name}v4.joblib')

['models/lightgbmv4.joblib']